In [1]:
import re
import json
import os
from pathlib import Path

def parse_aptitude_repo():
    # In a notebook, we get the current working directory
    # '..' moves up from the 'Notebooks' folder to the project root
    root_path = Path.cwd().parent 
    dataset = []
    
    # Regex to capture Question, Solution, and Answer
    q_pattern = r"\d+\.\s+\*\*Question\*\*:\s*(.*?)\s*\*\*Solution\*\*:\s*(.*?)\s*\*\*Answer\*\*:\s*(.*?)(?=\n\d+\.|\n#|\Z)"

    print(f"🔍 Searching for README files in: {root_path.resolve()}")

    # Recursively find all README.md files
    for readme_path in root_path.rglob("README.md"):
        
        # Skip the top-level README and any inside the Notebooks folder
        if readme_path.parent == root_path or "Notebooks" in readme_path.parts:
            continue
            
        # Extract Subject and Level from the folder names
        # Example: parts[-3] = Subject, parts[-2] = Level
        try:
            subject = readme_path.parts[-3]
            level = readme_path.parts[-2]
        except IndexError:
            subject, level = "General", "Unknown"

        with open(readme_path, 'r', encoding='utf-8') as f:
            content = f.read()

        # Split content by ## Category Headers
        sections = re.split(r"(##\s+.+?\n)", content)
        current_category = "General"

        for i in range(len(sections)):
            part = sections[i].strip()
            if not part: continue
            
            if part.startswith("##"):
                current_category = part.replace("##", "").strip()
            else:
                matches = re.findall(q_pattern, part, re.DOTALL)
                
                for q_text, sol_text, ans_text in matches:
                    # Clean and format the data
                    user_content = (
                        f"Subject: {subject}\n"
                        f"Level: {level}\n"
                        f"Category: {current_category}\n"
                        f"Question: {q_text.strip()}"
                    )
                    
                    # We combine Solution and Answer for the Assistant response
                    assistant_content = f"Solution: {sol_text.strip()} Answer: {ans_text.strip()}"
                    
                    dataset.append({
                        "messages": [
                            {"role": "user", "content": user_content},
                            {"role": "assistant", "content": assistant_content}
                        ]
                    })

    return dataset



In [2]:

# --- Execution Section ---

# 1. Parse the data
all_questions = parse_aptitude_repo()

# 2. Save the output inside the Notebooks folder
output_filename = "smollm2_aptitude_train.jsonl"
with open(output_filename, "w", encoding="utf-8") as f:
    for entry in all_questions:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"--- Extraction Complete ---")
print(f"✅ Total Questions Parsed: {len(all_questions)}")
print(f"📁 Dataset saved to: {os.getcwd()}/{output_filename}")

# 3. Quick Preview of the first item
if all_questions:
    print("\n--- Sample Entry ---")
    print(json.dumps(all_questions[0], indent=2, ensure_ascii=False))

🔍 Searching for README files in: D:\Others\ETL\ETL-CSE-Aptitude-Test-Practice-Hub
--- Extraction Complete ---
✅ Total Questions Parsed: 1769
📁 Dataset saved to: D:\Others\ETL\ETL-CSE-Aptitude-Test-Practice-Hub\Notebooks/smollm2_aptitude_train.jsonl

--- Sample Entry ---
{
  "messages": [
    {
      "role": "user",
      "content": "Subject: 06 Technical Aptitude (Basic Programming and AIML Concepts)\nLevel: 01 Basic\nCategory: Programming Fundamentals (1–20)\nQuestion: What is the output of this C code: `int x = 5; printf(\"%d\", x++);`?"
    },
    {
      "role": "assistant",
      "content": "Solution: `x++` is post-increment, so `printf` uses the current value of `x` (5), then increments `x` to 6. Answer: 5"
    }
  ]
}


In [3]:
## 🚀 Push to Hugging Face
from datasets import load_dataset
from huggingface_hub import login

# 1. Login (it will ask for your token)
login()

# 2. Load the JSONL file
dataset = load_dataset("json", data_files="./smollm2_aptitude_train.jsonl")

# 3. Push to Hub
# Change 'your-username' to your actual HF username
dataset.push_to_hub("Prathamesh25/aptitude-qa-dataset")

print("Done! Your dataset is now live.")


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Done! Your dataset is now live.


In [7]:
! pip install ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/914.9 kB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 914.9/914.9 kB 1.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 1.7 MB/s eta 0:00:01
   -------------- ------------------------- 0.8/2.2 MB 1.7 MB/s eta 0:00:01
   ----------------------- ---------------- 1.3/2.2 MB 2.0 MB/s eta 0:00:01
   --------------------------------- ------ 1.8/2.2 MB 2.1 MB/s eta 0:00:01
   -------------------------------------- - 2.1/2.2 MB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 1.7 MB/s eta 0:00:00

   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ----------


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
